# 运动损伤风险预测与干预建议

## 课程作业：数据挖掘实践

本项目旨在通过机器学习方法预测运动员的损伤风险，并基于预测结果提供个性化的干预建议。

## 1. 导入必要的库

In [ ]:
# 基础数据处理库
import numpy as np
import pandas as pd

# 数据可视化库
import matplotlib.pyplot as plt
import seaborn as sns

# 机器学习库
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# 设置随机种子确保结果可复现
np.random.seed(42)

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

## 2. 生成模拟数据集

### 数据集说明
- **样本量**: 200名运动员 × 30条记录 = 6000条样本
- **特征变量**:
  - `athlete_id`: 运动员编号
  - `training_duration`: 训练时长（分钟）
  - `movement_type`: 动作类型（慢跑、快跑、冲刺、跳跃、力量训练）
  - `heart_rate`: 心率（次/分钟）
- **目标变量**:
  - `injury`: 损伤标签（1=损伤，0=未损伤）

### 损伤概率规则
1. 训练时长 > 120分钟：损伤概率 +20%
2. 动作类型为冲刺或跳跃：损伤概率 +30%
3. 心率 > 170：损伤概率 +25%
4. 基础损伤概率：10%

In [ ]:
# 生成模拟数据集
def generate_simulated_data():
    """
    生成运动损伤模拟数据集
    
    返回:
        DataFrame: 包含6000条记录的数据集
    """
    # 参数设置
    n_athletes = 200  # 运动员数量
    n_records_per_athlete = 30  # 每位运动员的记录数
    
    # 动作类型列表
    movement_types = ['慢跑', '快跑', '冲刺', '跳跃', '力量训练']
    
    # 初始化数据列表
    data = []
    
    for athlete_id in range(1, n_athletes + 1):
        for _ in range(n_records_per_athlete):
            # 生成训练时长：50-180分钟，偏正态分布
            training_duration = int(np.random.normal(90, 30))
            training_duration = max(50, min(180, training_duration))
            
            # 生成动作类型：均匀分布
            movement_type = np.random.choice(movement_types)
            
            # 生成心率：根据动作类型有所差异
            base_heart_rate = {
                '慢跑': 120,
                '快跑': 140,
                '冲刺': 175,
                '跳跃': 150,
                '力量训练': 110
            }
            heart_rate = int(np.random.normal(base_heart_rate[movement_type], 15))
            heart_rate = max(60, min(200, heart_rate))
            
            # 计算损伤概率
            base_prob = 0.1  # 基础损伤概率10%
            
            # 规则1: 训练时长>120分钟，概率+20%
            if training_duration > 120:
                base_prob += 0.2
            
            # 规则2: 冲刺或跳跃，概率+30%
            if movement_type in ['冲刺', '跳跃']:
                base_prob += 0.3
            
            # 规则3: 心率>170，概率+25%
            if heart_rate > 170:
                base_prob += 0.25
            
            # 确保概率在合理范围内
            injury_prob = min(0.95, max(0.02, base_prob))
            
            # 根据概率生成损伤标签
            injury = 1 if np.random.random() < injury_prob else 0
            
            data.append({
                'athlete_id': athlete_id,
                'training_duration': training_duration,
                'movement_type': movement_type,
                'heart_rate': heart_rate,
                'injury': injury
            })
    
    return pd.DataFrame(data)

# 生成数据集
df = generate_simulated_data()

# 查看数据集基本信息
print("数据集形状:", df.shape)
print("\n数据集前5行:")
display(df.head())

# 查看损伤标签分布
print("\n损伤标签分布:")
print(df['injury'].value_counts(normalize=True))

## 3. 数据预处理

### 处理步骤
1. **独热编码**: 对动作类型进行独热编码
2. **标准化**: 对数值特征（训练时长、心率）进行标准化
3. **划分数据集**: 按照8:2比例划分为训练集和测试集

In [ ]:
# 分离特征和目标变量
X = df.drop(['injury', 'athlete_id'], axis=1)
y = df['injury']

# 划分训练集和测试集（8:2）
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 保存原始特征名称（用于后续特征重要性分析）
numeric_features = ['training_duration', 'heart_rate']
categorical_features = ['movement_type']

# 对数值特征进行标准化
scaler = StandardScaler()
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

# 对分类特征进行独热编码
encoder = OneHotEncoder(drop='first', sparse=False, handle_unknown='ignore')
encoded_train = encoder.fit_transform(X_train[categorical_features])
encoded_test = encoder.transform(X_test[categorical_features])

# 创建编码后的特征名称
encoded_feature_names = encoder.get_feature_names_out(categorical_features)

# 将编码后的特征转换为DataFrame
encoded_train_df = pd.DataFrame(encoded_train, columns=encoded_feature_names, index=X_train.index)
encoded_test_df = pd.DataFrame(encoded_test, columns=encoded_feature_names, index=X_test.index)

# 合并数值特征和编码后的分类特征
X_train_processed = pd.concat([X_train[numeric_features], encoded_train_df], axis=1)
X_test_processed = pd.concat([X_test[numeric_features], encoded_test_df], axis=1)

# 查看处理后的数据集
print("处理后的训练集形状:", X_train_processed.shape)
print("\n处理后的训练集前5行:")
display(X_train_processed.head())

## 4. 模型训练与评估

### 训练两个分类模型
1. **逻辑回归**: 经典的线性分类模型，可解释性强
2. **随机森林**: 集成学习模型，处理非线性关系能力强

### 评估指标
- 分类报告（精确率、召回率、F1-score）
- AUC-ROC
- 混淆矩阵热力图

In [ ]:
# 定义模型评估函数
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    """
    训练并评估模型
    
    参数:
        model: 机器学习模型
        X_train: 训练集特征
        y_train: 训练集标签
        X_test: 测试集特征
        y_test: 测试集标签
        model_name: 模型名称
    
    返回:
        model: 训练好的模型
        y_pred: 预测结果
        y_proba: 预测概率
        auc: AUC-ROC分数
    """
    # 训练模型
    model.fit(X_train, y_train)
    
    # 预测
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    # 计算AUC
    auc = roc_auc_score(y_test, y_proba)
    
    # 输出分类报告
    print(f"=== {model_name} 分类报告 ===")
    print(classification_report(y_test, y_pred))
    print(f"AUC-ROC: {auc:.4f}")
    
    # 绘制混淆矩阵热力图
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['未损伤', '损伤'],
                yticklabels=['未损伤', '损伤'])
    plt.title(f'{model_name} 混淆矩阵')
    plt.xlabel('预测标签')
    plt.ylabel('真实标签')
    plt.show()
    
    return model, y_pred, y_proba, auc

# 训练逻辑回归模型
log_reg = LogisticRegression(random_state=42, max_iter=200)
log_reg, y_pred_lr, y_proba_lr, auc_lr = evaluate_model(
    log_reg, X_train_processed, y_train, X_test_processed, y_test, '逻辑回归'
)

# 训练随机森林模型
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)
rf, y_pred_rf, y_proba_rf, auc_rf = evaluate_model(
    rf, X_train_processed, y_train, X_test_processed, y_test, '随机森林'
)

## 5. 特征重要性分析

### 随机森林特征重要性
随机森林模型可以输出每个特征对预测的重要程度，这有助于我们理解哪些因素对运动损伤风险影响最大。

In [ ]:
# 获取特征重要性
feature_importance = pd.DataFrame({
    'feature': X_train_processed.columns,
    'importance': rf.feature_importances_
})

# 按重要性排序
feature_importance = feature_importance.sort_values(by='importance', ascending=False).reset_index(drop=True)

# 绘制特征重要性条形图
plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=feature_importance, palette='viridis')
plt.title('随机森林特征重要性')
plt.xlabel('重要性得分')
plt.ylabel('特征名称')
plt.show()

# 输出特征重要性表格
print("特征重要性排名:")
display(feature_importance)

## 6. 前3个重要特征解释

基于随机森林的特征重要性分析，我们可以看到影响运动损伤风险的前三大因素：

### 1. 心率 (heart_rate)
- **重要性**: 最高
- **解释**: 心率是反映运动员身体负荷的直接指标。当心率超过170次/分钟时，运动员处于高强度训练状态，身体疲劳累积加快，受伤风险显著增加。持续高心率训练会导致心肌耗氧量剧增，肌肉过度疲劳，反应速度下降。

### 2. 训练时长 (training_duration)
- **重要性**: 第二
- **解释**: 训练时长直接影响身体疲劳程度。超过120分钟的长时间训练会导致：
  - 肌肉糖原耗尽
  - 核心温度升高
  - 注意力下降
  - 动作技术变形

### 3. 冲刺动作 (movement_type_冲刺)
- **重要性**: 第三
- **解释**: 冲刺是爆发力极强的动作，对肌肉、韧带和关节都有巨大冲击力。冲刺时：
  - 肌肉瞬间承受超过体重数倍的力量
  - 膝关节、踝关节承受巨大剪切力
  - 加速和减速过程易导致拉伤

## 7. 损伤风险预测与干预建议函数

编写一个实用函数，输入运动员的训练数据，输出损伤概率和个性化干预建议。

In [ ]:
def predict_injury_risk(duration, heart_rate, movement_type):
    """
    预测运动员损伤风险并提供干预建议
    
    参数:
        duration: 训练时长（分钟）
        heart_rate: 心率（次/分钟）
        movement_type: 动作类型（慢跑、快跑、冲刺、跳跃、力量训练）
    
    返回:
        dict: 包含损伤概率和干预建议
    """
    # 创建输入数据DataFrame
    input_data = pd.DataFrame({
        'training_duration': [duration],
        'heart_rate': [heart_rate],
        'movement_type': [movement_type]
    })
    
    # 数据预处理（与训练时一致）
    input_data[numeric_features] = scaler.transform(input_data[numeric_features])
    encoded_input = encoder.transform(input_data[categorical_features])
    encoded_input_df = pd.DataFrame(encoded_input, columns=encoded_feature_names)
    input_processed = pd.concat([input_data[numeric_features].reset_index(drop=True), 
                                 encoded_input_df], axis=1)
    
    # 预测损伤概率
    risk_prob = rf.predict_proba(input_processed)[:, 1][0]
    
    # 生成干预建议
    suggestions = []
    
    # 根据心率给出建议
    if heart_rate > 170:
        suggestions.append(f"⚠️ 心率过高({heart_rate}次/分)，建议立即降低训练强度，进行2-3分钟的active recovery（动态恢复），如慢走或深呼吸。")
    elif heart_rate > 150:
        suggestions.append(f"⚡ 心率较高({heart_rate}次/分)，建议每10分钟进行1分钟的补水和短暂休息。")
    
    # 根据训练时长给出建议
    if duration > 120:
        suggestions.append(f"⏰ 训练时长过长({duration}分钟)，建议将训练拆分为多个45-60分钟的单元，中间至少休息15分钟。")
    elif duration > 90:
        suggestions.append(f"⏳ 训练时长适中({duration}分钟)，建议在训练中后段增加5分钟的拉伸放松时间。")
    
    # 根据动作类型给出建议
    high_risk_movements = ['冲刺', '跳跃']
    if movement_type in high_risk_movements:
        suggestions.append(f"🏃 {movement_type}属于高风险动作，建议：1) 确保充分热身（至少10分钟动态拉伸）；2) 控制动作次数在合理范围；3) 训练后进行冰敷处理。")
    else:
        suggestions.append(f"🏋️ {movement_type}相对安全，建议保持正确的动作姿势，注意呼吸节奏。")
    
    # 如果风险概率特别高，增加额外建议
    if risk_prob > 0.7:
        suggestions.append("🚨 损伤风险极高！建议立即停止训练，进行全面的身体检查，并考虑安排至少24小时的休息恢复。")
    elif risk_prob > 0.5:
        suggestions.append("⚠️ 损伤风险较高，建议密切关注身体感受，如有不适立即停止。")
    
    # 确保返回3条建议
    while len(suggestions) < 3:
        suggestions.append("💡 建议定期进行身体机能检测，建立个人训练负荷档案。")
    
    return {
        'risk_probability': round(risk_prob * 100, 2),
        'risk_level': get_risk_level(risk_prob),
        'suggestions': suggestions[:3]
        # 取前3条建议
    }

def get_risk_level(prob):
    """
    根据概率返回风险等级
    """
    if prob < 0.2:
        return '低风险'
    elif prob < 0.4:
        return '中低风险'
    elif prob < 0.6:
        return '中高风险'
    else:
        return '高风险'

# 测试预测函数
print("=== 测试案例1：高风险场景 ===")
result1 = predict_injury_risk(duration=150, heart_rate=185, movement_type='冲刺')
print(f"损伤概率: {result1['risk_probability']}%")
print(f"风险等级: {result1['risk_level']}")
print("干预建议:")
for i, suggestion in enumerate(result1['suggestions'], 1):
    print(f"  {i}. {suggestion}")

print("\n=== 测试案例2：中等风险场景 ===")
result2 = predict_injury_risk(duration=90, heart_rate=145, movement_type='快跑')
print(f"损伤概率: {result2['risk_probability']}%")
print(f"风险等级: {result2['risk_level']}")
print("干预建议:")
for i, suggestion in enumerate(result2['suggestions'], 1):
    print(f"  {i}. {suggestion}")

print("\n=== 测试案例3：低风险场景 ===")
result3 = predict_injury_risk(duration=60, heart_rate=110, movement_type='慢跑')
print(f"损伤概率: {result3['risk_probability']}%")
print(f"风险等级: {result3['risk_level']}")
print("干预建议:")
for i, suggestion in enumerate(result3['suggestions'], 1):
    print(f"  {i}. {suggestion}")

## 8. 技术路线说明

### 整体流程图
```
数据生成 → 数据预处理 → 模型训练 → 模型评估 → 特征分析 → 预测应用
```

### 详细技术路线

#### 阶段1：数据生成
- **目标**: 创建符合业务规则的模拟数据集
- **方法**: 
  - 基于真实运动场景设计特征变量
  - 根据领域知识定义损伤概率规则
  - 使用NumPy生成符合统计分布的模拟数据
- **输出**: 6000条记录的结构化数据集

#### 阶段2：数据预处理
- **目标**: 将原始数据转换为模型可接受的格式
- **方法**:
  - **独热编码**: 使用sklearn.OneHotEncoder处理分类特征（动作类型）
  - **标准化**: 使用sklearn.StandardScaler对数值特征进行Z-score标准化
  - **数据集划分**: 使用train_test_split按8:2比例划分，保持类别分布(stratify)
- **输出**: 训练集(4800条)和测试集(1200条)

#### 阶段3：模型训练
- **目标**: 构建并训练分类模型
- **方法**:
  - **逻辑回归**: 作为基准模型，提供线性可解释性
  - **随机森林**: 作为提升模型，处理非线性关系
- **参数设置**:
  - 逻辑回归: max_iter=200确保收敛
  - 随机森林: n_estimators=100, max_depth=10, class_weight='balanced'
- **输出**: 训练好的模型对象

#### 阶段4：模型评估
- **目标**: 全面评估模型性能
- **方法**:
  - **分类报告**: 精确率(Precision)、召回率(Recall)、F1-score
  - **AUC-ROC**: 评估模型区分能力
  - **混淆矩阵**: 可视化分类结果
- **输出**: 模型性能指标和可视化图表

#### 阶段5：特征重要性分析
- **目标**: 理解影响损伤风险的关键因素
- **方法**: 
  - 提取随机森林的feature_importances_
  - 可视化条形图展示特征重要性排名
- **输出**: 特征重要性排名和领域解释

#### 阶段6：预测应用
- **目标**: 开发实用的预测工具
- **方法**:
  - 封装预测逻辑为函数
  - 根据输入特征生成个性化干预建议
  - 根据风险概率分级输出建议
- **输出**: 可直接调用的预测函数

### 关键技术要点

1. **数据质量保证**:
   - 设置随机种子确保结果可复现
   - 数据标准化消除量纲差异
   - 类别不平衡处理（class_weight='balanced'）

2. **模型选择策略**:
   - 逻辑回归作为基准，确保基线性能
   - 随机森林作为提升模型，捕捉非线性关系

3. **可解释性设计**:
   - 特征重要性分析揭示关键影响因素
   - 基于规则的干预建议生成机制

4. **工程化考虑**:
   - 数据预处理管道与训练时保持一致
   - 函数封装便于后续集成
   - 清晰的文档和注释

## 9. 总结

本项目完成了一个完整的运动损伤风险预测系统，主要成果包括：

### 已完成的工作
1. ✅ **数据生成**: 创建了6000条模拟数据，包含200名运动员的训练记录
2. ✅ **数据预处理**: 实现了独热编码和标准化，划分了训练/测试集
3. ✅ **模型训练**: 训练了逻辑回归和随机森林两个分类模型
4. ✅ **模型评估**: 输出了分类报告、AUC和混淆矩阵热力图
5. ✅ **特征分析**: 展示了随机森林特征重要性，解释了前3个重要特征
6. ✅ **预测函数**: 实现了损伤概率预测和个性化干预建议生成
7. ✅ **技术路线**: 提供了完整的技术路线说明

### 模型性能对比
- **逻辑回归**: 简单高效，可解释性强，适合作为基准
- **随机森林**: 性能更优，能捕捉复杂非线性关系

### 实际应用价值
- 帮助教练实时监测运动员状态
- 提供个性化训练建议
- 提前预警潜在损伤风险
- 优化训练计划，降低损伤率